# Manual Quality Checks and Preprocessing Configuration

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 13, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 3.5 Sonnet  

---

This notebook prepares the pre-processing steps that require manual input on a subject-level:

1. Checking the integrity of the recorded EEG data.
2. Selection of channels for interpolation.
3. Analyzing ICA results and selecting independent components for exclusion.

The manual information is logged in a pandas DataFrame, which is used as input in the subsequent automated preprocessing notebook (`3_preprocessing.ipynb`). This allows the pipeline to loop over a selection (or all) of the participants.

[] Should I crop the data before channel selection?

### 1. Setup, Configuration, and Behavioral Exclusion Check

This initialization cell prepares the workspace for the current participant. Specifically, it:

* Imports libraries and enables interactive plotting (`%matplotlib qt`), which is required for manual EEG/ICA inspection.
* Sets the current subject ID and locates the BIDS and derivatives directories.
* Loads the existing `preprocessing_config.csv` metadata table or creates a new one if this is the first run.
* Cross-references the subject with `subject_exclusions.json`. If the participant failed behavioral criteria,     
it automatically logs their exclusion in the config file and throws a warning so you know to skip the rest of the notebook.

In [1]:
import mne
import pandas as pd
import json
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
import matplotlib

# Sets the matplotlib backend to open a separate interactive window
%matplotlib qt 

# ==========================================
# 1.1. DEFINE CURRENT SUBJECT
# ==========================================
# Change this ID for every participant you want to inspect
subj = "22" # Today (13.07.) I will run 17 - DONE, 20 - Messy data!, 21, 22, 23 - DONE

# ==========================================
# 1.2. DEFINE PATHS & LOAD EXCLUSIONS DUE TO BEHAVIORAL RESULTS
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"

# Load json object with subjects that should be entirely excluded from the analysis
exclusion_file = derivatives_dir / "subject_exclusions.json"

# The JSON object is loaded into exclusions, a Python dictionary
with open(exclusion_file, "r") as f:
    exclusions = json.load(f)

# Use .get() to safely load the list of values, defaulting to an empty list if the key is missing
behavioral_exclusions = exclusions.get("behavioral_exclusions", [])

print("The following subjects are marked for exclusion based on behavioral criteria:")
print(behavioral_exclusions)

# Path to the central configuration file
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 3. LOAD OR CREATE CONFIGURATION FILE
# ==========================================
if config_path.exists():
    df_config = pd.read_csv(config_path, dtype={'subject': str})
    print(f"\nExisting config loaded. {len(df_config)} subjects inspected so far.")
else:
    df_config = pd.DataFrame(columns=[
        "subject", 
        "is_excluded", 
        "eeg_enc_recorded", 
        "eeg_ret_recorded", 
        "beh_enc_complete", 
        "beh_ret_complete",
        "bad_channels", 
        "bad_icas", 
        "notes_exclusion", 
        "notes_beh", 
        "notes_eeg"
    ])
    print("\nNo existing config found. Creating a new detailed metadata table.")

# ==========================================
# 4. SAFETY CHECK & AUTO-LOGGING
# ==========================================
# Check if the current subject is in the exclusion list
subject_is_excluded = subj in behavioral_exclusions

if subject_is_excluded:
    # 4a. Throw a highly visible warning
    print("\n" + "!"*65)
    print(f" WARNING: Subject {subj} is in the behavioral_exclusions list!")
    print("!"*65)
    
    # 4b. Automatically log this into the config dataframe
    # NOTE: since ICA/channel inspection never runs for auto-excluded subjects,
    # bad_channels/bad_icas/notes_eeg are left empty here.
    auto_notes = "Auto-excluded: Behavioral criteria not met."
    
    if subj in df_config['subject'].values:
        idx = df_config.index[df_config['subject'] == subj].tolist()[0]
        df_config.loc[idx, 'is_excluded'] = True
        df_config.loc[idx, 'notes_exclusion'] = auto_notes
    else:
        new_row = pd.DataFrame([{
            "subject": subj, 
            "is_excluded": True,
            "eeg_enc_recorded": pd.NA,
            "eeg_ret_recorded": pd.NA,
            "beh_enc_complete": pd.NA,
            "beh_ret_complete": pd.NA,
            "bad_channels": "", 
            "bad_icas": "",
            "ica_notes": "",
            "notes_exclusion": auto_notes,
            "notes_beh": "",
            "notes_eeg": ""
        }])
        df_config = pd.concat([df_config, new_row], ignore_index=True)
        
    # 4c. Save immediately
    df_config.to_csv(config_path, index=False)
    print(f"\n--> Action taken: Subject {subj} automatically marked as excluded in the config.")
    print("--> You can SKIP the rest of the cells in this notebook for this subject.")

else:
    print(f"\n--> Subject {subj} is clear for preprocessing. Proceed to the next cells.")

The following subjects are marked for exclusion based on behavioral criteria:
['05', '06', '07', '08', '13', '14', '16', '18', '19', '24', '25', '28', '31', '34', '39', '40']

Existing config loaded. 7 subjects inspected so far.

--> Subject 22 is clear for preprocessing. Proceed to the next cells.


### 2. Checking the integrity of the recorded EEG data

The following two plotting cells allow me to assess the completeness of the EEG recordings:

* Have triggers been successfully recorded and mapped as expected?
* Do we have signal recordings from 64 channels throughout the whole experimental procedure?

In [4]:
# ==========================================
# 2.1 PLOT EVENT MARKERS
# ==========================================

# Set BIDS path
bids_path = BIDSPath(
    subject=subj, 
    task='loc', 
    datatype='eeg', 
    root=bids_root)

# Load Raw
raw_eeg, bids_event_id = read_raw_bids(
    bids_path=bids_path, 
    return_event_dict=True,  # Extracts events mapping from BIDS (previous botebook)
    verbose='error'
)
raw_eeg.load_data()

# Load events
events, event_id = mne.events_from_annotations(
    raw_eeg, 
    event_id=bids_event_id,  
    verbose=False
)

# Plot event markers as function of experiment run time
fig_ev = mne.viz.plot_events(
    events, 
    sfreq=raw_eeg.info["sfreq"], 
    first_samp=raw_eeg.first_samp, 
    event_id=event_id
)

Reading 0 ... 2562019  =      0.000 ...  2562.019 secs...


C:\Users\noahm\AppData\Local\Temp\ipykernel_1108\2272340068.py:28: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  fig_ev = mne.viz.plot_events(


In [5]:
# ==========================================
# 2.2. PLOT THE RAW, CONTINUOUS DATA
# ==========================================
print(f"Inspecting subject: {subj}")

# Create a copy with basic filters for easier visual inspection
# (1Hz highpass removes slow drifts, 50 notch removes line noise)
raw_inspect = raw_eeg.copy().filter(l_freq=1.0, h_freq=None).notch_filter(50) 

# Plot time series
# Obviously bad channels could already be marked at this stage by clicking on them.
raw_inspect.plot(
    duration=10, 
    n_channels=32, 
    scalings=dict(eeg=20e-6),
    theme="light" 
)

print(f"Bad channels marked during inspection: {raw_inspect.info['bads']}")

Inspecting subject: 22
Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Fil

Channels marked as bad:
none


### 2. Selection of channels for interpolation.

While some bad channels might have been documented during the recording or in the previous cell, we will now create some more diagnostic plots to select channels that will need to be interpolated.

#### 2.1  Power Spectrum Density (PSD) plot
The PSD plot tells us the **power**  of each channel in logarithmic scale (y-axis) across a continuous range of frequencies (x-axis). The underlyig computation is a **Fourier transformation**, in which each channel's time-series data is decomposed into sine waves of varying frequencies. In this plot, the power refers to the squared amplitude of those sine waves at a given frequency.  

This plot can help detect channels that require interpolation. Normal data will show a 1/f decay curve and an alpha peak around 8 to 12 Hz. Red flags:
(a) Near-zero variance across all frequencies: no signal recorded, defect electrode
(b) Elevated power across most/all frequencies relevate to all other channels: muscular activity (EMG)

In [ ]:
# Run PSD plot command
# 'fmax' sets x-axis range from 0 to 100 Hz, allowing me to check for line noise
fig1 = raw_eeg.compute_psd(fmax=100).plot() # raw data (unfiltered)

fig2 = raw_inspect.compute_psd(fmax=100).plot() # 1Hz highpass, and 50 Hz notch filter

Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).


#### 2.1 Topoplots

The two topographical plots (for the encoding and retrieval phase) depict the average voltage at each channel across a number of events for a set of time points (-0.5, 0, 0.5, 1 seconds relative to fixation cross). We use fixation crosses to yield a high number of epochs.    

 Expected behavior: the brain's electrival activity should spread smoothly across the scalp due to volume conduction. Thus, we can flag channels if they stand out as isolated hot/cold spots relative to their neighbors. 

In [6]:
# 2. Define both events as list of tuples: (name, id)
target_events = [
    ('enc_fixation', 110),
    ('ret_fixation', 210),
]

for event_name, event_id in target_events:

    # 3. Extract events for this specific event only
    events, _ = mne.events_from_annotations(
        raw_inspect,
        event_id={event_name: event_id},
        verbose=False
    )

    # 4. Check if the target ID exists
    if event_id not in events[:, 2]:
        print(f"Warning: Event ID {event_id} ('{event_name}') not found. Skipping.")
        continue

    print(f"Plotting topomap for event: '{event_name}' (ID: {event_id})")

    # 5. Create Epochs
    epochs_viz = mne.Epochs(
        raw_inspect,
        events,
        event_id={event_name: event_id},
        tmin=-0.5, tmax=1,
        baseline=(None, 0),                     # Baseline correction for -0.5 to 0 s
        preload=True,                           # Epoch data is loaded into memory
        reject_by_annotation=True,              # BAD annotations from previous steps are considered
        verbose=False                       
    )

    # 6. Plot Topomap
    fig_topo = epochs_viz.average().plot_topomap(
        show_names=True,
        size=3,
        nrows = 1  # Zur Unterscheidung
    )

Plotting topomap for event: 'enc_fixation' (ID: 110)
Plotting topomap for event: 'ret_fixation' (ID: 210)


### 3. Analyzing ICA results and selecting independent components for exclusion

In the following cells, we run a preliminary Independent Component Analysis (ICA) and save the resulting unmixing matrices into the subjects' folder. The actual correction of the data based on the ICA computations happens during the batch processing in `3_preprocessing.ipynb`.

ICA is an unsupervised machine learning algorithm that can separate the EEG signal into statistically independent subcomponents that contribute to the recording signal: e.g., neural activity from the brain, eye blinks (EOG), heartbeats (ESC), muscle tension (EMG), channel noise.  

For computational efficiency and susceptibility to low-frequency noise, we run the ICA on an EEG copy that is downsampled, re-referenced to average, and filtered between 1 and 100 Hz. The ICA is then computed using the PICARD algorithm, and using every 3rd data point of the EEG recording.

In [7]:
# ==========================================
# 6. RUN INDEPENDENT COMPONENT ANALYSIS (ICA)
# ==========================================
import matplotlib
matplotlib.use('Agg')  # No interactive windows — just save figures to disk
import matplotlib.pyplot as plt

# Enter bad channels from previous inspection steps
raw_eeg.info['bads'] = ['FC2', 'T8', 'Fp2']

# We create a filtered, average-referenced, downsampled copy of the raw data for the preliminary ICA
raw_ica = raw_eeg.copy().filter(l_freq=1.0, h_freq=100) 
raw_ica.set_eeg_reference('average')
raw_ica.resample(250)  

# Rank refers to the dimensionality of the data. 64 channels - 1 (average reference) - n bad channels
rank = mne.compute_rank(raw_ica, tol='auto')['eeg']
print(f"Using n_components={rank} based on computed data rank.")

from mne.preprocessing import ICA
ica = ICA(
    n_components=rank, 
    method='picard', # Preconditioned ICA for Real Data: optimization algorithm
    fit_params=dict(ortho=False, extended=True),  # makes picard mathematically equivalent to Extended Infomax, which is widely considered the gold standard
    random_state=97,
    max_iter='auto' 
)
ica.fit(raw_ica, decim=3) # Since we have a very long recording, it's okay to only use every 3rd data point for ICA fitting

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 100.00 Hz
- Upper transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 112.50 Hz)
- Filter length: 3301 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Computing rank from data with rank=None
    Using tolerance 1.3e-10 (2.2e-16 eps * 60 dim * 9.8e+03  max singular value)
    Estimated rank (eeg): 59
    EEG: rank 59 computed from 60 data channels with 0 projectors
Using n_components=59 based on computed data rank.
Fitting ICA to data using 60 channels (please be patien

Method,picard
Fit parameters,ortho=Falseextended=Truemax_iter=500
Fit,94 iterations on raw data (213502 samples)
ICA components,59
Available PCA components,60
Channel types,eeg
ICA components marked for exclusion,—


After the ICA ran, we use our recordings from the physical EOG channels aswell as the `mne_icalabel` package to classify the extracted ICs: https://mne.tools/mne-icalabel/dev/generated/examples/00_iclabel.html   
This machine learning classifier assigns a probability that an IC belongs to one of several classes: e.g., brain, muscle, eye, heart, line noise, channel noise, other. The excluded ICs and their properties are saved as plots to the subject's derivatives folder and can be inspected there.

In [ ]:
# ==========================================
# 7a. EOG DETECTION + ICLABEL CLASSIFICATION
# ==========================================
eog_indices, eog_scores = ica.find_bads_eog(raw_ica) # Correlates ICs with EOG channel signals

from mne_icalabel import label_components
ic_labels = label_components(raw_ica, ica, method='iclabel')
labels = ic_labels['labels']
probs = ic_labels['y_pred_proba']

# Auto-exclude: any non-brain, non-other label with prob > 0.90, plus EOG-detected
auto_exclude = sorted(set(eog_indices) | {
    idx for idx, (label, prob) in enumerate(zip(labels, probs))
    if label not in ('brain', 'other') and prob > 0.90
})

print(f"\n--- Auto-excluded ICs (>90% confidence or EOG-detected) ---")
for idx in auto_exclude:
    print(f"IC{idx:02d}: {labels[idx]} (p={probs[idx]:.2f})")

confirmed_exclude_icas = auto_exclude

# ==========================================
# 7b. SAVE DIAGNOSTIC PLOTS TO DISK
# ==========================================
import os
subj_deriv_dir_ica = derivatives_dir / f"sub-{subj}" / "eeg" / "ica_qc"
subj_deriv_dir_ica.mkdir(parents=True, exist_ok=True)

# Overview of all components
fig_components = ica.plot_components(show=False)
if isinstance(fig_components, list):
    for i, f in enumerate(fig_components):
        f.savefig(os.path.join(subj_deriv_dir_ica, f"components_overview_{i}.png"), dpi=100)
        plt.close(f)
else:
    fig_components.savefig(os.path.join(subj_deriv_dir_ica, "components_overview.png"), dpi=100)
    plt.close(fig_components)

# Full diagnostic plots ONLY for excluded components
if confirmed_exclude_icas:
    figs = ica.plot_properties(raw_ica, picks=confirmed_exclude_icas, show=False)
    for idx, fig in zip(confirmed_exclude_icas, figs):
        fname = f"IC{idx:02d}_{labels[idx]}_{probs[idx]:.2f}.png"
        fig.savefig(os.path.join(subj_deriv_dir_ica, fname), dpi=100)
        plt.close(fig)

print(f"\nSaved QC plots to: {subj_deriv_dir_ica}")

# ==========================================
# 7d. SAVE FITTED ICA SOLUTION FOR NOTEBOOK 3
# ==========================================

# Saving the actually fitted ICA object (unmixing matrix) for Notebook 3
ica_path = subj_deriv_dir_ica / f"sub-{subj}_ica.fif"

# Store the confirmed exclusions on the object itself
ica.exclude = confirmed_exclude_icas

ica.save(ica_path, overwrite=True)
print(f"--> ICA solution saved to: {ica_path}")

Using EOG channels: HEOG, VEOG
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.2

In [ ]:
# ==========================================
# 8. LOG YOUR MANUAL DECISIONS & METADATA
# ==========================================

# Check if the subject is in your predefined full-exclusion list
subject_is_excluded = subj in behavioral_exclusions

# A. Phase Completion Flags 
# Manually change the Boolean flags if a specific phase is missing or corrupted
my_eeg_enc_recorded = True
my_eeg_ret_recorded = True
my_beh_enc_complete = True
my_beh_ret_complete = True

# B. Bad Channels & ICAs
my_bad_channels = raw_eeg.info['bads'] if not subject_is_excluded else []
# my_manual_icas = confirmed_exclude_icas if not subject_is_excluded else []

# B2. Build a short ICLabel justification string for each excluded IC
ica_reasons = ", ".join(
    f"IC{idx:02d}:{labels[idx]}({probs[idx]:.2f})"
    for idx in confirmed_exclude_icas
) if confirmed_exclude_icas else ""

# C. Targeted Notes
my_notes_exclusion = ""
my_notes_beh = ""
my_notes_eeg = ""
my_notes_ICA = f"ICA excluded: {ica_reasons}" if ica_reasons else ""

# ==========================================
# 9. FORMAT AND SAVE TO CSV
# ==========================================
bads_str = ", ".join(my_bad_channels) if my_bad_channels else ""
icas_str = ", ".join(map(str, confirmed_exclude_icas)) if confirmed_exclude_icas else ""

# Prepare the dictionary for the current subject
row_data = {
    "subject": subj, 
    "is_excluded": subject_is_excluded,
    "eeg_enc_recorded": my_eeg_enc_recorded,
    "eeg_ret_recorded": my_eeg_ret_recorded,
    "beh_enc_complete": my_beh_enc_complete,
    "beh_ret_complete": my_beh_ret_complete,
    "bad_channels": bads_str, 
    "bad_icas": icas_str,
    "ica_notes": my_notes_ICA,
    "notes_exclusion": my_notes_exclusion,
    "notes_beh": my_notes_beh,
    "notes_eeg": my_notes_eeg
}

if subj in df_config['subject'].values:
    # Update the existing row using the index
    idx = df_config.index[df_config['subject'] == subj].tolist()[0]
    for key, value in row_data.items():
        df_config.loc[idx, key] = value
    print(f"--> Metadata for sub-{subj} successfully updated.")
else:
    # Append a new row
    new_row = pd.DataFrame([row_data])
    df_config = pd.concat([df_config, new_row], ignore_index=True)
    print(f"--> New metadata entry for sub-{subj} successfully added.")

df_config.to_csv(config_path, index=False)
display(df_config.tail())

--> New metadata entry for sub-22 successfully added.


,subject,is_excluded,eeg_enc_recorded,eeg_ret_recorded,beh_enc_complete,beh_ret_complete,bad_channels,bad_icas,notes_exclusion,notes_beh,notes_eeg,ica_notes
2,24,True,NaN,NaN,NaN,NaN,NaN,NaN,Auto-excluded: Behavioral criteria not met.,NaN,NaN,NaN
3,26,False,True,True,True,True,"Fp2, Cz, AF4","0, 1, 10, 14, 25, 29, 34, 37",NaN,NaN,NaN,"ICA excluded: IC00:eye blink(0.99), IC01:eye b..."
4,17,False,True,True,True,True,"FC4, P2","1, 3, 5, 9, 12, 13, 18, 19, 21, 35, 37, 38",NaN,NaN,NaN,"ICA excluded: IC01:eye blink(0.28), IC03:chann..."
5,21,False,True,True,True,True,"PO8, P1, Pz","0, 1, 7, 10, 12, 16, 17, 20, 21, 24, 28, 33",NaN,NaN,NaN,"ICA excluded: IC00:eye blink(0.99), IC01:eye b..."
6,22,False,True,True,True,True,"FC2, T8, Fp2","1, 2, 3, 9, 13, 23, 24",,,,"ICA excluded: IC01:eye blink(1.00), IC02:eye b..."
